In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from geoai.utils_geo.raster_ops import RasterOperations
from geoai.utils_ds.dataframe_ops import DataFrameOperations


raster_ops = RasterOperations()
df_ops = DataFrameOperations()

In [2]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

In [3]:
df_bands = []
array = raster_ops.raster_to_array(RASTER_PATH)
raster_dimension = raster_ops.get_raster_dimensions(RASTER_PATH)


for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
    flat = raster_ops.flatten_array(array, band_index)
    df = df_ops.convert_to_df(flat, band_name)
    df = df.loc[~(df == 0).all(axis=1)]  # remove rows if all of its column is zero
    df_bands.append(df)
final_df_per_bands = pd.concat(df_bands, axis=1).astype('float64')
final_df_per_bands.head()


,BLUE,GREEN,RED,NIR,SWIR
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0


In [4]:
# Define the architecture of the neural network
class NNClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)  # input_dim is the number of input features
        self.fc2 = nn.Linear(16, 8)  # 16 neurons to 8 neurons
        self.fc3 = nn.Linear(8, output_dim) # 8 neurons to 4 neurons since we have 4 classes


    def forward(self, x):
        x = torch.relu(self.fc1(x)) # Apply ReLU activation function to the output of the first layer
        x = torch.relu(self.fc2(x)) # Apply ReLU activation function to the output of the second layer
        x = self.fc3(x) # Apply the output layer
        return x  # Return the output tensor
    
    # Note that we dont need to define the backward pass as PyTorch
    # automatically computes the gradients for us

In [5]:
input_dim = 5
output_dim = 4
trained_nn = NNClassifier(input_dim, output_dim)
trained_nn.load_state_dict(torch.load('trained_models/best_model_state_dict2.pth'))
trained_nn

C:\Users\Reginald\AppData\Local\Temp\ipykernel_12812\2355392302.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trained_nn.load_state_dict(torch.load('trained_models/bes

NNClassifier(
  (fc1): Linear(in_features=5, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=4, bias=True)
)

In [6]:
trained_nn.eval() # Set the model to evaluation mode

# Coverting the input data to a numpy array
data_np = final_df_per_bands.values.astype(np.float32)
data_tensor = torch.tensor(data_np)

# Predict
with torch.no_grad():
    outputs = trained_nn(data_tensor)
    _, predicted = torch.max(outputs, 1)
    predictions = predicted.cpu().numpy()

In [7]:
final_df_per_bands["PREDICTED_LC_NN"] = predictions
final_df_per_bands

,BLUE,GREEN,RED,NIR,SWIR,PREDICTED_LC_NN
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5,2
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5,2
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5,2
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5,2
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0,2
...,...,...,...,...,...,...
158219,1012.000000,971.333313,873.000000,2283.333252,2902.5,0
158220,1049.500000,1049.333374,961.000000,2343.000000,2677.0,1
158221,1046.000000,1049.333374,975.000000,2310.000000,2677.0,1
158222,765.333313,771.000000,722.000000,1962.500000,2168.5,1


In [8]:
raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC_NN2.tif",
    final_df_per_bands,
    "PREDICTED_LC_NN",
    raster_dimension,
    "float32",
)